In [1]:
from google.colab import files
import pandas as pd
import io

uploaded = files.upload()

print("\n=== 업로드 파일 ===")
for filename in uploaded.keys():
    print(filename)

Saving gangnam_db1_final_k2_cluster_result_2025.csv to gangnam_db1_final_k2_cluster_result_2025.csv
Saving gangnam_db1_region_behavior_fact_2025.csv to gangnam_db1_region_behavior_fact_2025.csv

=== 업로드 파일 ===
gangnam_db1_final_k2_cluster_result_2025.csv
gangnam_db1_region_behavior_fact_2025.csv


In [2]:
import pandas as pd

# =====================================================
# 1. 파일 불러오기
# =====================================================

cluster_file = "gangnam_db1_final_k2_cluster_result_2025.csv"
fact_file = "gangnam_db1_region_behavior_fact_2025.csv"

cluster_df = pd.read_csv(cluster_file)
fact_df = pd.read_csv(fact_file)


# =====================================================
# 2. 기본 구조 확인
# =====================================================

print("=== 분석 1: K=2 Cluster 결과 ===")
print("shape:", cluster_df.shape)
print("행정동 수:", cluster_df["행정동코드"].nunique())
print("행정동코드 중복:", cluster_df.duplicated(["행정동코드"]).sum())
print("결측:", cluster_df.isna().sum().sum())
print("\n컬럼:")
print(cluster_df.columns.tolist())


print("\n\n=== 분석 2: Behavior FACT ===")
print("shape:", fact_df.shape)
print("행정동 수:", fact_df["행정동코드"].nunique())
print("행정동코드 중복:", fact_df.duplicated(["행정동코드"]).sum())
print("결측:", fact_df.isna().sum().sum())
print("\n컬럼:")
print(fact_df.columns.tolist())


# =====================================================
# 3. 행정동 코드/이름 일치 여부
# =====================================================

cluster_keys = (
    cluster_df[
        ["행정동코드", "행정동명"]
    ]
    .drop_duplicates()
)

fact_keys = (
    fact_df[
        ["행정동코드", "행정동명"]
    ]
    .drop_duplicates()
)


key_check = cluster_keys.merge(
    fact_keys,
    on="행정동코드",
    how="outer",
    suffixes=("_분석1", "_분석2"),
    indicator=True
)


print("\n\n=== 행정동 KEY 검증 ===")
print(key_check.to_string(index=False))


print("\n분석1에만 존재:",
      (key_check["_merge"] == "left_only").sum())

print("분석2에만 존재:",
      (key_check["_merge"] == "right_only").sum())

both = key_check["_merge"] == "both"

print(
    "코드는 같지만 이름이 다른 행:",
    (
        key_check.loc[both, "행정동명_분석1"]
        !=
        key_check.loc[both, "행정동명_분석2"]
    ).sum()
)

=== 분석 1: K=2 Cluster 결과 ===
shape: (22, 25)
행정동 수: 22
행정동코드 중복: 0
결측: 0

컬럼:
['행정동코드', '행정동명', 'cluster_k2', 'total_flow_mean', 'flow_per_point_mean', 'age_20_ratio_mean', 'age_30_ratio_mean', 'age_50_ratio_mean', 'age_60_plus_ratio_mean', 'daytime_ratio_mean', 'morning_ratio_mean', 'very_low_outing_ratio_mean', 'very_low_outing_comm_ratio_mean', 'low_comm_ratio_mean', '평일 총 이동 횟수_mean', '집 추정 위치 평일 총 체류시간_mean', '평균 통화대상자 수_mean', '배달 서비스 사용일수_mean', 'foreigner_ratio_mean', 'resident_age_20_ratio', 'resident_age_60_plus_ratio', 'single_age_20_gap', 'single_age_30_gap', 'disabled_per_1000', 'welfare_vulnerability_pc1']


=== 분석 2: Behavior FACT ===
shape: (22, 10)
행정동 수: 22
행정동코드 중복: 0
결측: 13

컬럼:
['행정동코드', '행정동명', 'coherent_month_count', 'coherent_months', 'max_core_signal_count', 'avg_low_weekday_outing_signal', 'avg_very_low_outing_signal', 'avg_outside_activity_axis', 'avg_home_stay_axis', 'repeated_behavior_pattern']


=== 행정동 KEY 검증 ===
  행정동코드 행정동명_분석1 행정동명_분석2 _merge
1123051  

In [3]:
# =====================================================
# 1. coherent_months의 구조적 NaN 처리
# =====================================================

fact_df["coherent_months"] = (
    fact_df["coherent_months"]
    .fillna("")
)


# =====================================================
# 2. 분석 1의 지역유형만 추출
# =====================================================

cluster_key = cluster_df[
    [
        "행정동코드",
        "행정동명",
        "cluster_k2"
    ]
].copy()


# =====================================================
# 3. 분석 1 + 분석 2 결합
# =====================================================

integrated_df = fact_df.merge(
    cluster_key,
    on=[
        "행정동코드",
        "행정동명"
    ],
    how="left",
    validate="one_to_one"
)


# 보기 편하게 cluster를 앞쪽으로 이동
column_order = [
    "행정동코드",
    "행정동명",
    "cluster_k2",
    "coherent_month_count",
    "coherent_months",
    "max_core_signal_count",
    "avg_low_weekday_outing_signal",
    "avg_very_low_outing_signal",
    "avg_outside_activity_axis",
    "avg_home_stay_axis",
    "repeated_behavior_pattern"
]

integrated_df = integrated_df[
    column_order
]


# =====================================================
# 4. 결합 검증
# =====================================================

print("=== 분석 1 + 분석 2 결합 검증 ===")
print("shape:", integrated_df.shape)
print(
    "행정동 수:",
    integrated_df["행정동코드"].nunique()
)
print(
    "중복:",
    integrated_df
    .duplicated(["행정동코드"])
    .sum()
)
print(
    "전체 결측:",
    integrated_df.isna().sum().sum()
)


# =====================================================
# 5. Cluster별 행정동 수
# =====================================================

print("\n=== Cluster별 행정동 수 ===")

print(
    integrated_df["cluster_k2"]
    .value_counts()
    .sort_index()
)


# =====================================================
# 6. 22개 동 교차 결과 확인
# =====================================================

print("\n=== 지역유형 × 행동신호 ===")

print(
    integrated_df[
        [
            "행정동명",
            "cluster_k2",
            "coherent_month_count",
            "coherent_months",
            "max_core_signal_count",
            "repeated_behavior_pattern"
        ]
    ]
    .sort_values(
        [
            "cluster_k2",
            "coherent_month_count",
            "행정동명"
        ],
        ascending=[
            True,
            False,
            True
        ]
    )
    .to_string(index=False)
)

=== 분석 1 + 분석 2 결합 검증 ===
shape: (22, 11)
행정동 수: 22
중복: 0
전체 결측: 0

=== Cluster별 행정동 수 ===
cluster_k2
0    11
1    11
Name: count, dtype: int64

=== 지역유형 × 행동신호 ===
행정동명  cluster_k2  coherent_month_count           coherent_months  max_core_signal_count  repeated_behavior_pattern
 신사동           0                     2          2025-08, 2025-09                      4                          1
대치2동           0                     1                   2025-08                      2                          0
대치4동           0                     1                   2025-10                      3                          0
논현1동           0                     0                                                0                          0
논현2동           0                     0                                                1                          0
삼성1동           0                     0                                                2                          0
삼성2동           0              

In [4]:
import pandas as pd


# =====================================================
# 1. Cluster × coherent_month_count 교차표
# =====================================================

cross_count = pd.crosstab(
    integrated_df["cluster_k2"],
    integrated_df["coherent_month_count"]
)

print("=== Cluster × coherent_month_count ===")
print(cross_count)


# =====================================================
# 2. Cluster 내부 비율
# =====================================================

cross_ratio = pd.crosstab(
    integrated_df["cluster_k2"],
    integrated_df["coherent_month_count"],
    normalize="index"
) * 100

print("\n=== Cluster 내부 비율 (%) ===")
print(
    cross_ratio.round(1)
)


# =====================================================
# 3. Cluster별 행동신호 요약
# =====================================================

cluster_behavior_summary = (
    integrated_df
    .groupby("cluster_k2")
    .agg(
        dong_count=(
            "행정동코드",
            "count"
        ),

        coherent_month_mean=(
            "coherent_month_count",
            "mean"
        ),

        coherent_month_median=(
            "coherent_month_count",
            "median"
        ),

        coherent_month_max=(
            "coherent_month_count",
            "max"
        ),

        repeated_dong_count=(
            "repeated_behavior_pattern",
            "sum"
        ),

        max_core_signal_mean=(
            "max_core_signal_count",
            "mean"
        )
    )
    .reset_index()
)


cluster_behavior_summary[
    "repeated_dong_ratio"
] = (
    cluster_behavior_summary[
        "repeated_dong_count"
    ]
    /
    cluster_behavior_summary[
        "dong_count"
    ]
)


print("\n=== Cluster별 행동변화 요약 ===")
print(
    cluster_behavior_summary
    .round(3)
    .to_string(index=False)
)


# =====================================================
# 4. 반복 관찰 지역 확인
# =====================================================

print("\n=== 2개월 이상 반복 관찰 지역 ===")

print(
    integrated_df.loc[
        integrated_df[
            "coherent_month_count"
        ] >= 2,
        [
            "행정동명",
            "cluster_k2",
            "coherent_month_count",
            "coherent_months",
            "max_core_signal_count"
        ]
    ]
    .sort_values(
        [
            "cluster_k2",
            "coherent_month_count"
        ],
        ascending=[
            True,
            False
        ]
    )
    .to_string(index=False)
)

=== Cluster × coherent_month_count ===
coherent_month_count  0  1  2  3
cluster_k2                      
0                     8  2  1  0
1                     5  3  1  2

=== Cluster 내부 비율 (%) ===
coherent_month_count     0     1    2     3
cluster_k2                                 
0                     72.7  18.2  9.1   0.0
1                     45.5  27.3  9.1  18.2

=== Cluster별 행동변화 요약 ===
 cluster_k2  dong_count  coherent_month_mean  coherent_month_median  coherent_month_max  repeated_dong_count  max_core_signal_mean  repeated_dong_ratio
          0          11                0.364                    0.0                   2                    1                 1.455                0.091
          1          11                1.000                    1.0                   3                    3                 2.273                0.273

=== 2개월 이상 반복 관찰 지역 ===
행정동명  cluster_k2  coherent_month_count           coherent_months  max_core_signal_count
 신사동           0               

In [5]:
from google.colab import files
import pandas as pd

# =====================================================
# 1. CONTEXT 파일 업로드
# =====================================================

uploaded_context = files.upload()

Saving gangnam_db1_region_context_2025.csv to gangnam_db1_region_context_2025.csv


In [6]:
# =====================================================
# 2. CONTEXT 파일 불러오기
# =====================================================

context_file = "gangnam_db1_region_context_2025.csv"

context_df = pd.read_csv(
    context_file
)


# =====================================================
# 3. 기본 구조 검증
# =====================================================

print("=== CONTEXT 파일 검증 ===")

print(
    "shape:",
    context_df.shape
)

print(
    "행정동 수:",
    context_df["행정동코드"].nunique()
)

print(
    "행정동코드 중복:",
    context_df
    .duplicated(["행정동코드"])
    .sum()
)

print(
    "전체 결측:",
    context_df.isna().sum().sum()
)

print("\n=== 컬럼 ===")
print(
    context_df.columns.tolist()
)


# =====================================================
# 4. 기존 통합 데이터와 KEY 일치 확인
# =====================================================

key_check = (
    integrated_df[
        ["행정동코드", "행정동명"]
    ]
    .merge(
        context_df[
            ["행정동코드", "행정동명"]
        ],
        on="행정동코드",
        how="outer",
        suffixes=(
            "_통합",
            "_context"
        ),
        indicator=True
    )
)


print("\n=== KEY 검증 ===")

print(
    "통합 데이터에만 존재:",
    (key_check["_merge"] == "left_only").sum()
)

print(
    "CONTEXT에만 존재:",
    (key_check["_merge"] == "right_only").sum()
)

both = (
    key_check["_merge"] == "both"
)

print(
    "코드는 같지만 이름이 다른 행:",
    (
        key_check.loc[
            both,
            "행정동명_통합"
        ]
        !=
        key_check.loc[
            both,
            "행정동명_context"
        ]
    ).sum()
)

=== CONTEXT 파일 검증 ===
shape: (22, 36)
행정동 수: 22
행정동코드 중복: 0
전체 결측: 0

=== 컬럼 ===
['행정동코드', '행정동명', 'resident_total_mean', 'resident_total_mean_pct', 'foreigner_ratio_mean', 'foreigner_ratio_mean_pct', 'resident_age_20_ratio', 'resident_age_20_ratio_pct', 'resident_age_30_ratio', 'resident_age_30_ratio_pct', 'resident_age_40_ratio', 'resident_age_40_ratio_pct', 'resident_age_50_ratio', 'resident_age_50_ratio_pct', 'resident_age_60_plus_ratio', 'resident_age_60_plus_ratio_pct', 'single_age_20_gap', 'single_age_20_gap_pct', 'single_age_30_gap', 'single_age_30_gap_pct', 'single_age_40_gap', 'single_age_40_gap_pct', 'single_age_50_gap', 'single_age_50_gap_pct', 'single_age_60_plus_gap', 'single_age_60_plus_gap_pct', 'official_single_household_mean', 'official_single_household_mean_pct', 'official_single_household_change_rate', 'official_single_household_change_rate_pct', 'welfare_context_pc1', 'welfare_context_pc1_pct', 'disabled_total', 'disabled_total_pct', 'disabled_per_1000', 'disabled_

In [7]:
# =====================================================
# 1. integrated_df + CONTEXT 결합
# =====================================================

analysis_context = (
    integrated_df
    .merge(
        context_df.drop(
            columns=["행정동명"]
        ),
        on="행정동코드",
        how="left",
        validate="one_to_one"
    )
)


# =====================================================
# 2. 결합 검증
# =====================================================

print("=== 통합 Context 데이터 검증 ===")
print("shape:", analysis_context.shape)
print(
    "행정동 수:",
    analysis_context["행정동코드"].nunique()
)
print(
    "중복:",
    analysis_context
    .duplicated(["행정동코드"])
    .sum()
)
print(
    "결측:",
    analysis_context.isna().sum().sum()
)


# =====================================================
# 3. 행동관찰 그룹 생성
#
# 분석용 기술적 그룹이며
# 위험등급/라벨이 아님
# =====================================================

def observation_group(count):

    if count == 0:
        return "관찰 없음"

    elif count == 1:
        return "1개월 관찰"

    else:
        return "2개월 이상 반복"


analysis_context[
    "observation_group"
] = (
    analysis_context[
        "coherent_month_count"
    ]
    .apply(observation_group)
)


# =====================================================
# 4. Cluster 1만 추출
# =====================================================

cluster1_context = (
    analysis_context[
        analysis_context["cluster_k2"] == 1
    ]
    .copy()
)


print("\n=== Cluster 1 행동관찰 그룹 ===")

print(
    cluster1_context[
        "observation_group"
    ]
    .value_counts()
)


# =====================================================
# 5. 대표 Context 선택
#
# 중복되는 원값과 percentile을 모두 비교하지 않고
# 우선 해석하기 쉬운 percentile 중심으로 확인
# =====================================================

context_compare_cols = [

    # 주민 구조
    "resident_total_mean_pct",
    "resident_age_20_ratio_pct",
    "resident_age_30_ratio_pct",
    "resident_age_40_ratio_pct",
    "resident_age_50_ratio_pct",
    "resident_age_60_plus_ratio_pct",

    # 1인가구
    "single_age_20_gap_pct",
    "single_age_30_gap_pct",
    "single_age_40_gap_pct",
    "single_age_50_gap_pct",
    "single_age_60_plus_gap_pct",

    # 1인세대 규모/변화
    "official_single_household_mean_pct",
    "official_single_household_change_rate_pct",

    # 복지·경제
    "welfare_context_pc1_pct",

    # 장애
    "disabled_per_1000_pct"
]


# =====================================================
# 6. Cluster 1의 11개 동 실제 값 확인
# =====================================================

print("\n=== Cluster 1 : 동별 Context ===")

print(
    cluster1_context[
        [
            "행정동명",
            "observation_group",
            "coherent_month_count"
        ]
        + context_compare_cols
    ]
    .sort_values(
        [
            "coherent_month_count",
            "행정동명"
        ],
        ascending=[
            False,
            True
        ]
    )
    .round(1)
    .to_string(index=False)
)


# =====================================================
# 7. 그룹별 Context 중앙값
# =====================================================

group_order = [
    "관찰 없음",
    "1개월 관찰",
    "2개월 이상 반복"
]


context_group_median = (
    cluster1_context
    .groupby(
        "observation_group"
    )[context_compare_cols]
    .median()
    .reindex(group_order)
)


print("\n=== Cluster 1 : 그룹별 Context 중앙값 ===")

print(
    context_group_median
    .round(1)
    .T
    .to_string()
)

=== 통합 Context 데이터 검증 ===
shape: (22, 45)
행정동 수: 22
중복: 0
결측: 0

=== Cluster 1 행동관찰 그룹 ===
observation_group
관찰 없음        5
1개월 관찰       3
2개월 이상 반복    3
Name: count, dtype: int64

=== Cluster 1 : 동별 Context ===
행정동명 observation_group  coherent_month_count  resident_total_mean_pct  resident_age_20_ratio_pct  resident_age_30_ratio_pct  resident_age_40_ratio_pct  resident_age_50_ratio_pct  resident_age_60_plus_ratio_pct  single_age_20_gap_pct  single_age_30_gap_pct  single_age_40_gap_pct  single_age_50_gap_pct  single_age_60_plus_gap_pct  official_single_household_mean_pct  official_single_household_change_rate_pct  welfare_context_pc1_pct  disabled_per_1000_pct
개포3동         2개월 이상 반복                     3                     22.7                        9.1                       63.6                       54.5                       18.2                            90.9                   13.6                   13.6                   22.7                   81.8                        95.5  

In [8]:
# =====================================================
# Cluster 1 핵심 Context 후보 직접 비교
# =====================================================

focus_context = [
    "resident_total_mean_pct",
    "resident_age_60_plus_ratio_pct",
    "single_age_60_plus_gap_pct",
    "welfare_context_pc1_pct",
    "disabled_per_1000_pct"
]


focus_table = (
    cluster1_context[
        [
            "행정동명",
            "observation_group",
            "coherent_month_count"
        ]
        + focus_context
    ]
    .sort_values(
        [
            "coherent_month_count",
            "행정동명"
        ],
        ascending=[
            False,
            True
        ]
    )
)


print("=== Cluster 1 핵심 Context 후보 ===")

print(
    focus_table
    .round(1)
    .to_string(index=False)
)


# =====================================================
# 반복 3개 동 각각의 위치 확인
# =====================================================

print(
    "\n=== 반복 3개 동 개별 Context ==="
)

print(
    focus_table[
        focus_table[
            "coherent_month_count"
        ] >= 2
    ]
    .round(1)
    .to_string(index=False)
)


# =====================================================
# Context가 높은데 행동신호가 반복되지 않은 반례 확인
#
# 80 percentile은 설명을 위한 탐색선일 뿐
# 통계적 임계값이 아님
# =====================================================

counterexample = (
    cluster1_context[
        (
            cluster1_context[
                focus_context
            ] >= 80
        )
        .any(axis=1)
        &
        (
            cluster1_context[
                "coherent_month_count"
            ] < 2
        )
    ][
        [
            "행정동명",
            "observation_group",
            "coherent_month_count"
        ]
        + focus_context
    ]
)


print(
    "\n=== 높은 Context를 가지지만 반복관찰되지 않은 지역 ==="
)

print(
    counterexample
    .sort_values(
        "coherent_month_count",
        ascending=False
    )
    .round(1)
    .to_string(index=False)
)

=== Cluster 1 핵심 Context 후보 ===
행정동명 observation_group  coherent_month_count  resident_total_mean_pct  resident_age_60_plus_ratio_pct  single_age_60_plus_gap_pct  welfare_context_pc1_pct  disabled_per_1000_pct
개포3동         2개월 이상 반복                     3                     22.7                            90.9                        95.5                     90.9                   90.9
일원본동         2개월 이상 반복                     3                     45.5                            50.0                       100.0                     27.3                   81.8
일원1동         2개월 이상 반복                     2                     13.6                            95.5                        63.6                     95.5                   95.5
개포2동            1개월 관찰                     1                     95.5                            31.8                        86.4                     36.4                   22.7
개포4동            1개월 관찰                     1                     50.0         

In [9]:
# =====================================================
# 1. Cluster 0 추출
# =====================================================

cluster0_context = (
    analysis_context[
        analysis_context["cluster_k2"] == 0
    ]
    .copy()
)

print("=== Cluster 0 구성 ===")
print("행정동 수:", len(cluster0_context))

print(
    cluster0_context[
        [
            "행정동명",
            "coherent_month_count",
            "coherent_months"
        ]
    ]
    .sort_values(
        ["coherent_month_count", "행정동명"],
        ascending=[False, True]
    )
    .to_string(index=False)
)


# =====================================================
# 2. Cluster 0에서 확인할 대표 Context
#
# Cluster 1과 동일한 구조변수 +
# 신사동에서 특징적이었던 20·30대 1인가구 gap
# =====================================================

cluster0_focus = [
    "resident_total_mean_pct",
    "resident_age_20_ratio_pct",
    "resident_age_30_ratio_pct",
    "resident_age_60_plus_ratio_pct",

    "single_age_20_gap_pct",
    "single_age_30_gap_pct",
    "single_age_60_plus_gap_pct",

    "official_single_household_mean_pct",

    "welfare_context_pc1_pct",
    "disabled_per_1000_pct"
]


# =====================================================
# 3. Cluster 0 전체 Context 확인
# =====================================================

cluster0_table = (
    cluster0_context[
        [
            "행정동명",
            "coherent_month_count"
        ]
        + cluster0_focus
    ]
    .sort_values(
        ["coherent_month_count", "행정동명"],
        ascending=[False, True]
    )
)


print("\n=== Cluster 0 : 동별 Context ===")

print(
    cluster0_table
    .round(1)
    .to_string(index=False)
)


# =====================================================
# 4. 신사동의 Cluster 0 내부 percentile 계산
#
# 기존 *_pct = 강남구 22개 동 내 위치
# 아래 값 = Cluster 0의 11개 동 안에서의 위치
# =====================================================

cluster0_internal = (
    cluster0_context[
        ["행정동명"] + [
            col.replace("_pct", "")
            for col in cluster0_focus
        ]
    ]
    .copy()
)


raw_focus_cols = [
    col.replace("_pct", "")
    for col in cluster0_focus
]


for col in raw_focus_cols:
    cluster0_internal[
        f"{col}_cluster0_pct"
    ] = (
        cluster0_internal[col]
        .rank(
            pct=True,
            method="average"
        )
        * 100
    )


internal_pct_cols = [
    f"{col}_cluster0_pct"
    for col in raw_focus_cols
]


print(
    "\n=== 신사동 : Cluster 0 내부 상대위치 ==="
)

print(
    cluster0_internal.loc[
        cluster0_internal["행정동명"] == "신사동",
        ["행정동명"] + internal_pct_cols
    ]
    .round(1)
    .to_string(index=False)
)


# =====================================================
# 5. 신사동과 나머지 Cluster 0 중앙값 대조
#
# 통계검정이 아니라 기술적 비교
# =====================================================

sinsa = (
    cluster0_context[
        cluster0_context["행정동명"] == "신사동"
    ]
)

cluster0_others = (
    cluster0_context[
        cluster0_context["행정동명"] != "신사동"
    ]
)


comparison_rows = []

for col in cluster0_focus:

    comparison_rows.append(
        {
            "context": col,
            "신사동": sinsa[col].iloc[0],
            "Cluster0_나머지10개_중앙값":
                cluster0_others[col].median()
        }
    )


sinsa_comparison = pd.DataFrame(
    comparison_rows
)

sinsa_comparison[
    "차이"
] = (
    sinsa_comparison["신사동"]
    -
    sinsa_comparison[
        "Cluster0_나머지10개_중앙값"
    ]
)


print(
    "\n=== 신사동 vs Cluster 0 나머지 10개 동 ==="
)

print(
    sinsa_comparison
    .round(1)
    .to_string(index=False)
)

=== Cluster 0 구성 ===
행정동 수: 11
행정동명  coherent_month_count  coherent_months
 신사동                     2 2025-08, 2025-09
대치2동                     1          2025-08
대치4동                     1          2025-10
논현1동                     0                 
논현2동                     0                 
삼성1동                     0                 
삼성2동                     0                 
압구정동                     0                 
역삼1동                     0                 
역삼2동                     0                 
 청담동                     0                 

=== Cluster 0 : 동별 Context ===
행정동명  coherent_month_count  resident_total_mean_pct  resident_age_20_ratio_pct  resident_age_30_ratio_pct  resident_age_60_plus_ratio_pct  single_age_20_gap_pct  single_age_30_gap_pct  single_age_60_plus_gap_pct  official_single_household_mean_pct  welfare_context_pc1_pct  disabled_per_1000_pct
 신사동                     2                     18.2                       68.2                       36.4        